# 2. CORRMAP: finding artifact components in all participants

Follows `ana02_corrmap.m` of the reference pipeline. **Run once, after `01_ica.ipynb` was run for every participant** (and again when participants are added).

CORRMAP is semi-automatic:
1. You choose, by hand, one clear example ("template") of each artifact type: eye blink, heartbeat, lateral eye movement (as in the reference) and muscle (added: participants tap the whole time, and beta is one of our bands). Write them into `CORRMAP_TEMPLATES` in `config.py`.
2. CORRMAP finds the components with a similar scalp map (correlation ≥ 0.8) in every participant.
3. The found components are saved per participant and removed in `03_clean.ipynb`.

In [ ]:
import mne
import pandas as pd
import matplotlib.pyplot as plt
from mne.preprocessing import corrmap, read_ica

import config as cfg
%matplotlib inline

## 1. Loading all ICA solutions

All participants with a saved ICA (all sessions in `OUT_ROOT`). Their component plots are in each participant's folder (`sub-NNN_ica_components_*.png`).

In [ ]:
ica_files = sorted(cfg.OUT_ROOT.glob("sub-*/sub-*_ica.fif"))
subjects = [f.name.removesuffix("_ica.fif") for f in ica_files]
icas = [read_ica(f) for f in ica_files]
print(f"{len(icas)} participants:", subjects)

## 2. Choosing the templates

Look at the component plots and pick, for each artifact type, the participant and component that show it most clearly. Write them into `CORRMAP_TEMPLATES` in `config.py` (MNE counts components from 0).

- **Eye blink:** strong map over the forehead (both sides), same sign.
- **Lateral eye movement:** front left and front right with opposite signs.
- **Heartbeat:** a broad diagonal gradient across the head; regular spikes about once per second in the time course.
- **Muscle** (added to the reference): a small, focal spot at the edge of the head (temples, neck, jaw), often on one side; spiky, irregular time course; power that stays high above ~20 Hz instead of falling. If one template catches only one side, add a second one (`muscle_left`, `muscle_right`).

The next cell shows the chosen templates.

In [ ]:
import importlib
importlib.reload(cfg)  # pick up changes made to config.py

for label, (subject, component) in cfg.CORRMAP_TEMPLATES.items():
    figure = icas[subjects.index(subject)].plot_components(picks=[component], show=False)
    figure.suptitle(f"{label}: {subject}, component {component}")

## 3. Running CORRMAP

For each template, CORRMAP finds the matching components in every participant (threshold 0.8). It works in two rounds: first with the template, then with the average map of the components found in the first round.

In [ ]:
found = []
for label, (subject, component) in cfg.CORRMAP_TEMPLATES.items():
    corrmap(icas, template=(subjects.index(subject), component), threshold=cfg.CORRMAP_THRESHOLD,
            label=label, plot=True, show=True)
    for s, ica in zip(subjects, icas):
        for c in ica.labels_.get(label, []):
            found.append({"subject": s, "component": int(c), "reason": label})

found = pd.DataFrame(found, columns=["subject", "component", "reason"])
found.pivot_table(index="subject", columns="reason", values="component", aggfunc=lambda c: list(c))

## 4. Saving the components to remove

Per participant: the CORRMAP components plus any `EXTRA_EXCLUDE` from `config.py` (after visual review). A PNG with the removed components is saved for each participant (as `_badcomps.png` in `ana02`).

In [ ]:
summary = []
for s, ica in zip(subjects, icas):
    rows = found[found["subject"] == s]
    extra = pd.DataFrame({"subject": s, "component": cfg.EXTRA_EXCLUDE.get(s, []), "reason": "manual"})
    rows = pd.concat([rows, extra]).drop_duplicates("component")
    rows["component"] = rows["component"].astype(int)  # pandas may turn them into floats
    rows.to_csv(cfg.out_dir(s) / f"{s}_badcomps.csv", index=False)

    components = sorted(rows["component"].tolist())
    if components:
        figure = ica.plot_components(picks=components, show=False)
        figure.suptitle(f"{s} removed components: {components}")
        figure.savefig(cfg.out_dir(s) / f"{s}_badcomps.png", dpi=100)
        plt.close(figure)
    summary.append({"subject": s, "n_removed": len(components), "components": components})

pd.DataFrame(summary)